# Atlas CS17 -> ALL 87 structures -> exactly N balanced groups (default 4) -> nnU-Net datasets -> NEW Hugging Face repo

## Purpose

This CPU notebook builds **N_GROUPS nnU-Net training datasets** (default **4**) from Atlas's own
canonical grey volume + expert label volume (`3datlas.3dembryo.nl`). Every structure ID found in
the labels TIFF is kept -- none is filtered -- and each ID lands in exactly one group. Each group
becomes its own independent multiclass model.

**Atlas grey volume -> Atlas label volume -> measure every label ID in the pixels -> split IDs into
exactly `N_GROUPS` balanced groups -> one nnU-Net dataset per group -> fixed-seed train/held-out
slice split per group -> QA -> push to a BRAND-NEW Hugging Face dataset repo + manifest**

### What changed vs the previous version

1. **The group count is now chosen by you (`N_GROUPS = 4`), not by the data.** The old size-ratio
   rule produced 10 lopsided groups (3, 11, 12, 10, 11, 11, 11, 8, 9 and **1** class). Now you get
   exactly 4 groups with near-equal class counts (about 22 each for 87 structures).
2. **Two things are balanced at once:**
   * *class counts* -- every group has about `87 / 4` classes (within `COUNT_TOLERANCE`);
   * *class sizes inside each group* -- structures are sorted by pixel volume and cut into
     contiguous size tiers, so a group never mixes a giant structure with a tiny one. This is what
     stops one big class from dominating a model's loss. The cut points are found by exhaustive
     search (optimal for the score described in Section 4).
3. **A comparison table is printed** for the selected strategy and the alternative (`snake`), so you
   can see the balance numbers before anything is exported.
4. **New Hugging Face repos + safety guard.** Output goes to `senku111/atlas-cs17-4groups-dataset`.
   The notebook refuses to run if that repo already contains a `training_dataset/` folder or if
   it points at any of the old repos, so old and new data can never be mixed.
5. New nnU-Net dataset IDs (701-704) so nothing collides with the old 601-610 datasets.

Unchanged: no `.am` name file (a domain expert names the IDs later), global intensity normalization
shared by every group, and a physically separate held-out folder per group.


In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
from pathlib import Path
import os, sys, json, math, re, time, shutil, subprocess
import numpy as np
import pandas as pd

# --- grey + label volumes come directly from Atlas ---
ATLAS_BASE_URLS = [
    "http://3datlas.3dembryo.nl",
    "https://3datlas.3dembryo.nl",
]
STAGE = "CS17"

# ------------------------------------------------------------------------------------
# Atlas specimen code for CS17 (the part that was "3512" for CS15)
# ------------------------------------------------------------------------------------
# Every Atlas Carnegie stage uses its own specimen code in its filenames, e.g.
# 3DAtlas_CS15-3512-grey-v2016-01.tif, 3DAtlas_CS12-8505A-v2016-03.pdf, 3DAtlas_CS09-H712-...
# Verified directly from the Atlas's own "Atlas contents" download page (3dembryoatlas.com/blank,
# which links straight to 3datlas.3dembryo.nl): the Stage 17 (42-44 days) entry is filed under
# 3DAtlas_CS17-6521-v2016-03.pdf, i.e. specimen code "6521".
ATLAS_SPECIMEN_ID = "6521"  # CS17 specimen code, confirmed from the Atlas download page
if not ATLAS_SPECIMEN_ID:
    raise RuntimeError(
        "ATLAS_SPECIMEN_ID is not set. Open the CS17 model on the Atlas website, find its "
        "specimen code from the grey/label TIFF filename, and set ATLAS_SPECIMEN_ID above."
    )

SPECIMEN_ID = f"{STAGE}-{ATLAS_SPECIMEN_ID}"
PREFIX = f"3DAtlas_{STAGE}-{ATLAS_SPECIMEN_ID}"

# The Atlas has used more than one filename-version suffix across stages (v2016-01 for CS15,
# v2016-03 for CS09/CS12, etc). Rather than guess which one CS17 uses, every candidate is
# actually tried against the real server in Section 2 below and whichever one exists wins --
# this is a real existence check against the live server, not a guess about the data itself.
ATLAS_VERSION_CANDIDATES = ["v2016-01", "v2016-02", "v2016-03"]
ATLAS_SOURCE_FILES = {
    "grey_tif":   f"{PREFIX}-grey-{{version}}.tif",
    "labels_tif": f"{PREFIX}-labels-{{version}}.tif",
    # NOTE: the .am material-name file is intentionally NOT downloaded or used anywhere in
    # this notebook. Every label ID that exists as pixels in labels_tif is a training target,
    # named or not -- a domain expert names each ID later, from the trained models' predictions.
}

# --- where THIS notebook's own output (the training datasets) gets pushed to ---
HF_REPO_ID = "senku111/atlas-cs17-4groups-dataset"   # NEW repo -- never the old "3d-module-new"
HF_REPO_TYPE = "dataset"

# --- base dataset numbering: group 0 -> BASE_DATASET_ID, group 1 -> BASE_DATASET_ID+1, ... ---
BASE_DATASET_ID = 701   # old run used 601-610; new run uses 701-704
DATASET_NAME_PREFIX = "AtlasCS17FourGroup"

ROOT = Path(f"/kaggle/working/dataset_prep_{SPECIMEN_ID.replace('-', '_')}")
RAW_DIR = ROOT / "raw"
EXPORT_DIR = ROOT / "export"
REPORT_DIR = ROOT / "reports"
for d in [RAW_DIR, EXPORT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Balanced grouping -- WHY this exists
# ------------------------------------------------------------
# An embryo atlas has dozens of segmented structures of wildly different sizes. One multiclass
# model over all of them lets the biggest structures dominate the loss and starve the small ones.
# So the structures are split into N_GROUPS smaller multiclass models. Two goals, both enforced:
#   (a) every group has about the same NUMBER of classes (87 / 4 -> ~22 each), and
#   (b) inside a group the classes have SIMILAR SIZES, so none dominates the others.
# "balanced_contiguous" sorts structures by pixel volume and cuts the list into N_GROUPS
# contiguous size tiers (exhaustive search over cut points, optimal for the score in Section 4).
# "snake" is the alternative: equal counts AND equal total volume per group, but every group
# mixes giant and tiny classes -- worse for the dominance problem, kept only for comparison.
N_GROUPS = 4                                 # <-- how many modules / models you want
GROUPING_STRATEGY = "balanced_contiguous"    # or "snake"
COUNT_TOLERANCE = 3    # each group may hold n/N_GROUPS +/- this many classes (0 = exactly equal counts)
COUNT_PENALTY = 0.15   # extra cost per class of count spread; higher = counts closer to equal

# --- safety: this notebook must never write into the old repos ---
OLD_REPOS = ["senku111/3d-module-new", "senku111/atlas-cs17-nnunet", "senku111/atlas-cs17-nnunet-training-v3-new"]
assert HF_REPO_ID not in OLD_REPOS, f"HF_REPO_ID points at an OLD repo: {HF_REPO_ID}"

# --- fixed-seed train/held-out slice split, applied independently per group ---
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.20

print("Grey volume source: Atlas, direct download")
print("Label volume source: Atlas, direct download (expert ground-truth annotation)")
print(f"Class selection: NONE -- every label ID present in the TIFF is trained on, split")
print(f"across exactly {N_GROUPS} balanced groups (see Section 4).")
print("Output repo:", HF_REPO_ID)


Grey volume source: Atlas, direct download
Label volume source: Atlas, direct download (expert ground-truth annotation)
Class selection: NONE -- every label ID present in the TIFF is trained on, split
across exactly 4 balanced groups (see Section 4).
Output repo: senku111/atlas-cs17-4groups-dataset


## 2. Download the Atlas grey + label volumes

Both files come from the same source, same request pattern. The Atlas TIFFs are
LZW-compressed; a known-compatible `tifffile`/`imagecodecs` pair is installed before import.
The `.am` material file is **not** downloaded.


In [2]:
# ============================================================
# 2. AUTOMATIC ATLAS DOWNLOAD + LOAD
# ============================================================
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
    "tifffile==2023.7.10", "imagecodecs==2024.6.1",
])
import tifffile

def download_atlas_file(filename, retries=4):
    dest = RAW_DIR / filename
    if dest.exists() and dest.stat().st_size > 1024:
        print(f"Already present: {filename} ({dest.stat().st_size/1024**2:.1f} MB)")
        return dest
    last_error = None
    for base in ATLAS_BASE_URLS:
        url = f"{base}/{filename}"
        for attempt in range(1, retries + 1):
            try:
                print(f"Downloading {url} [attempt {attempt}/{retries}]")
                import urllib.request
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 Atlas CS17 preparation"})
                with urllib.request.urlopen(req, timeout=900) as response, open(dest, "wb") as f:
                    shutil.copyfileobj(response, f)
                if dest.stat().st_size > 1024:
                    print(f"Saved: {dest} ({dest.stat().st_size/1024**2:.1f} MB)")
                    return dest
                raise RuntimeError("Downloaded file is unexpectedly small.")
            except Exception as e:
                last_error = e
                if dest.exists():
                    dest.unlink()
                print("  failed:", e)
                time.sleep(2 * attempt)
    raise RuntimeError(f"[DATA PROBLEM] could not download {filename}: {last_error}")

def resolve_versioned_filename(template):
    """Try each ATLAS_VERSION_CANDIDATES suffix with a lightweight HEAD request against the
    real server and return the first filename that actually exists there. This checks
    existence on the live Atlas server -- it never guesses at the DATA, only at which of a
    few previously-seen filename-version suffixes this specimen happens to use."""
    import urllib.request
    last_error = None
    for version in ATLAS_VERSION_CANDIDATES:
        filename = template.format(version=version)
        for base in ATLAS_BASE_URLS:
            url = f"{base}/{filename}"
            try:
                req = urllib.request.Request(url, method="HEAD",
                                              headers={"User-Agent": "Mozilla/5.0 Atlas CS17 preparation"})
                with urllib.request.urlopen(req, timeout=60) as response:
                    if response.status < 400:
                        print(f"Found on server: {filename}")
                        return filename
            except Exception as e:
                last_error = e
    raise RuntimeError(
        f"[DATA PROBLEM] none of {ATLAS_VERSION_CANDIDATES} exist for template '{template}' on "
        f"{ATLAS_BASE_URLS}. Double-check ATLAS_SPECIMEN_ID='{ATLAS_SPECIMEN_ID}' is correct, and "
        f"add the real version suffix to ATLAS_VERSION_CANDIDATES if it's a new one. "
        f"Last error: {last_error}"
    )

grey_filename = resolve_versioned_filename(ATLAS_SOURCE_FILES["grey_tif"])
labels_filename = resolve_versioned_filename(ATLAS_SOURCE_FILES["labels_tif"])
grey_tif_path = download_atlas_file(grey_filename)
labels_tif_path = download_atlas_file(labels_filename)

grey = tifffile.imread(grey_tif_path)
labels = tifffile.imread(labels_tif_path)
print("grey volume shape:", grey.shape)
print("labels volume shape:", labels.shape)
assert grey.shape == labels.shape, \
    f"[DATA PROBLEM] grey shape {grey.shape} != labels shape {labels.shape} -- " \
    "the Atlas grey volume and label volume must correspond slice-for-slice."


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 5.7 MB/s eta 0:00:00
Found on server: 3DAtlas_CS17-6521-grey-v2016-01.tif
Found on server: 3DAtlas_CS17-6521-labels-v2016-01.tif
Saved: /kaggle/working/dataset_prep_CS17_6521/raw/3DAtlas_CS17-6521-grey-v2016-01.tif (1244.7 MB)
Saved: /kaggle/working/dataset_prep_CS17_6521/raw/3DAtlas_CS17-6521-labels-v2016-01.tif (24.1 MB)
grey volume shape: (1059, 1821, 1126)
labels volume shape: (1059, 1821, 1126)


## 3. Measure representation of EVERY ID present in the TIFF

No `.am` file, no names, no filtering. For every non-background pixel value that actually
occurs in the label volume: slices present, slice coverage %, median positive area, total
positive pixels. Every one of these IDs will end up in some group's training set.


In [3]:
# ============================================================
# 3. REPRESENTATION ANALYSIS (every unique non-background pixel value in the volume)
# ============================================================
label_values = np.unique(labels).astype(np.int64)
label_values = label_values[label_values != 0]  # drop background
n_slices = labels.shape[0]
slice_area = labels.shape[-2] * labels.shape[-1]

print(f"{len(label_values)} distinct non-background structure IDs found in {labels_tif_path.name}")

rows = []
for label_id in label_values:
    label_id = int(label_id)
    per_slice_positive = (labels == label_id).reshape(n_slices, -1).sum(axis=1)
    slices_present = int((per_slice_positive > 0).sum())
    positive_slices = per_slice_positive[per_slice_positive > 0]

    rows.append({
        "atlas_label_id": label_id,
        "slices_present": slices_present,
        "slice_coverage_pct": 100.0 * slices_present / n_slices,
        "median_positive_area_px": float(np.median(positive_slices)) if len(positive_slices) else 0.0,
        "median_positive_area_pct": float(np.median(positive_slices)) / slice_area * 100.0 if len(positive_slices) else 0.0,
        "total_positive_pixels": int(per_slice_positive.sum()),
    })

stats_df = pd.DataFrame(rows).sort_values("total_positive_pixels", ascending=False).reset_index(drop=True)
stats_df.to_csv(REPORT_DIR / "class_representation_stats.csv", index=False)
print(f"All {len(stats_df)} structures kept -- nothing is filtered out or dropped.")
stats_df.head(20)


87 distinct non-background structure IDs found in 3DAtlas_CS17-6521-labels-v2016-01.tif
All 87 structures kept -- nothing is filtered out or dropped.


,atlas_label_id,slices_present,slice_coverage_pct,median_positive_area_px,median_positive_area_pct,total_positive_pixels
0,1,1056,99.716714,346920.0,16.919246,371991320
1,76,1046,98.772427,4830.5,0.235583,107492663
2,45,1050,99.150142,49048.0,2.392065,79118614
3,14,173,16.336166,67311.0,3.282749,17217176
4,64,854,80.642115,6366.5,0.310493,10687760
5,28,166,15.675165,60076.5,2.929924,9442654
6,30,358,33.805477,12193.5,0.594675,8819210
7,56,302,28.517469,22265.0,1.085861,8304104
8,43,820,77.431539,3274.5,0.159697,8008355
9,58,136,12.842304,54153.0,2.641035,6818017


## 4. Balanced grouping: split every ID into exactly `N_GROUPS` groups

Fully automatic, but the group count is yours. The selected strategy and the alternative are both
printed with their balance metrics:

* `n_classes` -- classes in the group (should be nearly equal across groups)
* `intra_group_size_ratio` -- largest / smallest class inside the group (lower = fairer model)
* `largest_class_share_pct` -- how much of the group's pixels ONE class owns (lower = less dominance)


In [4]:
# ============================================================
# 4. BALANCED GROUPING INTO EXACTLY N_GROUPS GROUPS
# ============================================================
import itertools

size_by_id = dict(zip(stats_df["atlas_label_id"], stats_df["total_positive_pixels"]))
sorted_ids = stats_df.sort_values("total_positive_pixels", ascending=False)["atlas_label_id"].tolist()
n_ids = len(sorted_ids)
assert N_GROUPS >= 1 and n_ids >= N_GROUPS, f"cannot make {N_GROUPS} groups from {n_ids} structures"

def _log_size(a):
    return math.log(max(int(size_by_id[a]), 1))

def group_stats(groups):
    """Balance metrics for a candidate grouping."""
    grand_total = sum(size_by_id[a] for g in groups for a in g)
    rows = []
    for gi, g in enumerate(groups):
        sizes = [int(size_by_id[a]) for a in g]
        rows.append({
            "group_index": gi,
            "n_classes": len(g),
            "total_positive_pixels": int(sum(sizes)),
            "pct_of_all_pixels": 100.0 * sum(sizes) / grand_total,
            "smallest_id_pixels": min(sizes),
            "largest_id_pixels": max(sizes),
            "intra_group_size_ratio": max(sizes) / max(min(sizes), 1),
            # how much of the group's total pixel volume ONE class owns (dominance inside the model)
            "largest_class_share_pct": 100.0 * max(sizes) / sum(sizes),
        })
    return pd.DataFrame(rows)

def group_contiguous(ids_sorted_desc, n_groups, tol, penalty):
    """Cut the size-sorted list into n_groups CONTIGUOUS blocks. Every group holds classes of
    similar size (lowest possible intra-group size ratio) and the class counts stay within
    +/- tol of n/n_groups. Exhaustive search over all cut points, so it is optimal for:
        score = worst intra-group log(size ratio) + penalty * (max count - min count)"""
    n = len(ids_sorted_desc)
    logs = [_log_size(a) for a in ids_sorted_desc]
    lo = max(1, n // n_groups - tol)
    hi = math.ceil(n / n_groups) + tol
    best_score, best_bounds = None, None
    for cuts in itertools.combinations(range(1, n), n_groups - 1):
        b = (0,) + cuts + (n,)
        counts = [b[i + 1] - b[i] for i in range(n_groups)]
        if min(counts) < lo or max(counts) > hi:
            continue
        worst = max(logs[b[i]] - logs[b[i + 1] - 1] for i in range(n_groups))
        score = worst + penalty * (max(counts) - min(counts))
        if best_score is None or score < best_score:
            best_score, best_bounds = score, b
    assert best_bounds is not None, "no valid split found -- increase COUNT_TOLERANCE"
    return [ids_sorted_desc[best_bounds[i]:best_bounds[i + 1]] for i in range(n_groups)]

def group_snake(ids_sorted_desc, n_groups):
    """Deal classes out largest-first in a back-and-forth (snake) order. Class counts are equal
    (+/-1) AND total pixel volume per group is nearly equal, but every group mixes giant and
    tiny structures (large intra-group size ratio)."""
    groups = [[] for _ in range(n_groups)]
    for i, a in enumerate(ids_sorted_desc):
        r, pos = divmod(i, n_groups)
        groups[pos if r % 2 == 0 else n_groups - 1 - pos].append(a)
    return groups

candidates = {
    "balanced_contiguous": group_contiguous(sorted_ids, N_GROUPS, COUNT_TOLERANCE, COUNT_PENALTY),
    "snake": group_snake(sorted_ids, N_GROUPS),
}
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
for name, groups in candidates.items():
    print(f"\n--- strategy: {name}" + ("   <== SELECTED" if name == GROUPING_STRATEGY else ""))
    print(group_stats(groups).to_string(index=False))

assert GROUPING_STRATEGY in candidates, f"GROUPING_STRATEGY must be one of {list(candidates)}"
final_groups = candidates[GROUPING_STRATEGY]
assert len(final_groups) == N_GROUPS

groups_summary_df = group_stats(final_groups)
groups_summary_df["atlas_ids"] = [sorted(int(a) for a in g) for g in final_groups]
groups_summary_df.to_csv(REPORT_DIR / "groups_summary.csv", index=False)
groups_summary = groups_summary_df.to_dict("records")

n_ids_covered = sum(len(g["atlas_ids"]) for g in groups_summary)
assert n_ids_covered == n_ids, f"[DATA PROBLEM] {n_ids_covered} IDs across groups != {n_ids} IDs found in the volume"
assert set().union(*[set(g["atlas_ids"]) for g in groups_summary]) == set(int(a) for a in sorted_ids), \
    "[DATA PROBLEM] grouping does not cover exactly the same ID set found in the volume"
print(f"\nVerified: every one of the {n_ids} structure IDs is in exactly one of the {N_GROUPS} groups "
      f"({', '.join(str(len(g)) for g in final_groups)} classes). None dropped, none duplicated.")



--- strategy: balanced_contiguous   <== SELECTED
 group_index  n_classes  total_positive_pixels  pct_of_all_pixels  smallest_id_pixels  largest_id_pixels  intra_group_size_ratio  largest_class_share_pct
           0         21              661515608               97.4             1235452          371991320                   301.1                     56.2
           1         22               14150813                2.1              320861            1196490                     3.7                      8.5
           2         22                3243596                0.5               60759             314448                     5.2                      9.7
           3         22                 540441                0.1                1054              59805                    56.7                     11.1

--- strategy: snake
 group_index  n_classes  total_positive_pixels  pct_of_all_pixels  smallest_id_pixels  largest_id_pixels  intra_group_size_ratio  largest_class_share_pct
     

## 5. Global intensity normalization (computed once, shared by every group)

One percentile range computed across the ENTIRE grey volume, applied identically to every
slice in every group's dataset -- not a per-slice or per-group range. This keeps a slice's
absolute brightness meaning the same thing everywhere.


In [5]:
# ============================================================
# 5. GLOBAL INTENSITY NORMALIZATION (computed once, shared by every group)
# ============================================================
from PIL import Image

# --- estimate the global percentile range from a random SUBSAMPLE, not the whole volume ---
# np.percentile(grey.astype(np.float32), ...) on the full volume allocates a full-size float32
# copy of `grey` (4 bytes/voxel). On a volume this large that copy alone can be several GB, and
# it happens on top of `grey` + `labels` already being resident -- that's what was hitting the
# 30 GB ceiling. A random subsample of a few tens of millions of voxels gives a statistically
# indistinguishable 0.5/99.5 percentile estimate at a tiny fraction of the memory cost.
rng = np.random.default_rng(SPLIT_SEED)
n_sample = min(grey.size, 20_000_000)
sample_idx = rng.integers(0, grey.size, size=n_sample)
grey_sample = grey.reshape(-1)[sample_idx].astype(np.float32)
global_lo, global_hi = np.percentile(grey_sample, [0.5, 99.5])
del grey_sample, sample_idx
print(f"Global intensity range used for ALL groups (from a {n_sample:,}-voxel sample): "
      f"low={global_lo:.2f}  high={global_hi:.2f}")

# --- normalize SLICE BY SLICE into a preallocated uint8 array -- never a full-volume float copy ---
norm_grey_stack = np.empty(grey.shape, dtype=np.uint8)
denom = float(global_hi - global_lo)
for z in range(grey.shape[0]):
    if denom <= 0:
        norm_grey_stack[z] = 0
        continue
    sl = grey[z].astype(np.float32)          # one slice's worth of float32, not the whole volume
    sl = np.clip((sl - global_lo) / denom, 0, 1)
    norm_grey_stack[z] = (sl * 255).astype(np.uint8)

print("Normalized grey stack shape:", norm_grey_stack.shape, norm_grey_stack.dtype)

Global intensity range used for ALL groups (from a 20,000,000-voxel sample): low=101.00  high=255.00
Normalized grey stack shape: (1059, 1821, 1126) uint8


## 6. Build, export, QA and push ONE nnU-Net dataset PER GROUP

For each group: remap that group's Atlas IDs to consecutive nnU-Net IDs (`0=background`,
`1..k`=that group's structures, placeholder-named `structure_<atlas_id>` since no names exist
yet), decide a fixed-seed train/held-out **slice** split independently for that group, export
`imagesTr`/`labelsTr` + a physically separate `heldout/` folder, run the same leak-proof QA as
before, then push that group's dataset to its own folder on Hugging Face.

A doctor names each `structure_<atlas_id>` later, once the trained models are predicting —
`class_mapping.json` (written per group) is exactly what makes that renaming traceable back
to the original Atlas IDs.


In [6]:
# ============================================================
# 6. BUILD, EXPORT, QA AND PUSH ONE DATASET PER GROUP
# ============================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
from huggingface_hub import HfApi

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        from getpass import getpass
        HF_TOKEN = getpass("Enter your Hugging Face WRITE token: ")

api = HfApi(token=HF_TOKEN)

# --- NEW-repo guard: create it if needed, and refuse to continue if it already holds a dataset ---
assert HF_REPO_ID not in OLD_REPOS, f"HF_REPO_ID points at an OLD repo: {HF_REPO_ID}"
api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, private=True, exist_ok=True)
_existing = [f for f in api.list_repo_files(HF_REPO_ID, repo_type=HF_REPO_TYPE) if f.startswith("training_dataset/")]
assert not _existing, (
    f"{HF_REPO_ID} already contains {len(_existing)} file(s) under training_dataset/. "
    "Use a fresh repo name in Section 1 so old and new data are never mixed.")
print("Fresh HF dataset repo confirmed empty:", HF_REPO_ID)

all_groups_manifest = {
    "specimen_id": SPECIMEN_ID,
    "grey_source": ATLAS_SOURCE_FILES["grey_tif"],
    "label_source": ATLAS_SOURCE_FILES["labels_tif"],
    "note": (
        "Every non-background label ID present in the Atlas labels TIFF was kept -- nothing "
        "was filtered by name, size, or quality score. IDs were split into the groups below "
        "purely to keep per-model class-size imbalance bounded. Each group is its own "
        "independent nnU-Net dataset/model. Class names are placeholders (structure_<atlas_id>) "
        "until a domain expert names each ID from the trained models' predictions."
    ),
    "n_groups": N_GROUPS,
    "grouping_strategy": GROUPING_STRATEGY,
    "count_tolerance": COUNT_TOLERANCE,
    "split_seed": SPLIT_SEED,
    "holdout_fraction": HOLDOUT_FRACTION,
    "normalization": {
        "method": "global percentile (0.5-99.5) over the entire grey volume, shared by every group",
        "global_lo": float(global_lo),
        "global_hi": float(global_hi),
    },
    "groups": [],
}

for group_index, atlas_ids in enumerate(final_groups):
    dataset_id = BASE_DATASET_ID + group_index
    dataset_name = f"{DATASET_NAME_PREFIX}{group_index:02d}"
    print(f"\n{'='*70}\nGROUP {group_index}  ->  Dataset{dataset_id:03d}_{dataset_name}  "
          f"({len(atlas_ids)} classes)\n{'='*70}")

    # --- remap this group's atlas IDs to consecutive nnU-Net IDs (0 = background) ---
    sorted_group_ids = sorted(int(a) for a in atlas_ids)
    atlas_to_nnunet = {0: 0}
    nnunet_to_name = {0: "background"}
    for new_id, atlas_id in enumerate(sorted_group_ids, start=1):
        atlas_to_nnunet[atlas_id] = new_id
        nnunet_to_name[new_id] = f"structure_{atlas_id}"

    remap_lut = np.zeros(int(labels.max()) + 1, dtype=np.uint8)
    for atlas_id, new_id in atlas_to_nnunet.items():
        remap_lut[atlas_id] = new_id
    group_mask = remap_lut[labels]

    # --- QA: shapes match, only expected IDs present, every selected class survived remap ---
    assert grey.shape == group_mask.shape, \
        f"[DATA PROBLEM] group {group_index}: grey shape {grey.shape} != mask shape {group_mask.shape}"
    expected_ids = set(range(len(sorted_group_ids) + 1))
    actual_ids = set(np.unique(group_mask).tolist())
    unexpected = actual_ids - expected_ids
    assert not unexpected, f"[DATA PROBLEM] group {group_index}: unexpected mask IDs: {unexpected}"
    for new_id, name in nnunet_to_name.items():
        if new_id == 0:
            continue
        assert (group_mask == new_id).any(), \
            f"[DATA PROBLEM] group {group_index}: class {name} vanished after remapping"

    # --- fixed-seed train/held-out SLICE split, decided before any files are written ---
    nonempty_slices = [z for z in range(group_mask.shape[0]) if group_mask[z].max() > 0]
    case_ids = [f"{SPECIMEN_ID.replace('-', '_')}_{z:05d}" for z in nonempty_slices]
    case_id_to_z = dict(zip(case_ids, nonempty_slices))

    rng_shuffled = pd.Series(case_ids).sample(frac=1.0, random_state=SPLIT_SEED).tolist()
    n_holdout = int(len(rng_shuffled) * HOLDOUT_FRACTION)
    holdout_cases = sorted(rng_shuffled[:n_holdout])
    train_cases = sorted(rng_shuffled[n_holdout:])
    print(f"{len(nonempty_slices)} non-empty slices -> train={len(train_cases)}, holdout={len(holdout_cases)}")

    # --- export: train -> imagesTr/labelsTr, held-out -> physically separate heldout/ ---
    dataset_root = EXPORT_DIR / f"Dataset{dataset_id:03d}_{dataset_name}"
    images_dir = dataset_root / "imagesTr"
    labels_dir = dataset_root / "labelsTr"
    heldout_images_dir = dataset_root / "heldout" / "images"
    heldout_labels_dir = dataset_root / "heldout" / "labels"
    for d in [images_dir, labels_dir, heldout_images_dir, heldout_labels_dir]:
        d.mkdir(parents=True, exist_ok=True)

    for case_id in train_cases:
        z = case_id_to_z[case_id]
        Image.fromarray(norm_grey_stack[z]).save(images_dir / f"{case_id}_0000.png")
        Image.fromarray(group_mask[z]).save(labels_dir / f"{case_id}.png")
    for case_id in holdout_cases:
        z = case_id_to_z[case_id]
        Image.fromarray(norm_grey_stack[z]).save(heldout_images_dir / f"{case_id}_0000.png")
        Image.fromarray(group_mask[z]).save(heldout_labels_dir / f"{case_id}.png")

    tr_file_stems = {p.stem.replace("_0000", "") for p in images_dir.glob("*.png")}
    overlap = tr_file_stems & set(holdout_cases)
    assert not overlap, f"[DATA LEAK] group {group_index}: held-out case(s) found in imagesTr: {overlap}"
    print(f"Verified: 0 of {len(holdout_cases)} held-out cases present in imagesTr/labelsTr.")

    split_manifest = {
        "seed": SPLIT_SEED, "holdout_fraction": HOLDOUT_FRACTION,
        "n_total": len(case_ids), "n_train": len(train_cases), "n_holdout": len(holdout_cases),
        "train_cases": train_cases, "holdout_cases": holdout_cases,
        "note": ("Held-out cases are held-out SLICES from this one specimen, not a held-out "
                 "specimen, and are stored in heldout/images + heldout/labels, physically "
                 "separate from imagesTr/labelsTr, so nnU-Net's own planning/preprocessing/"
                 "cross-validation never sees them."),
    }
    with open(dataset_root / "split_manifest.json", "w") as f:
        json.dump(split_manifest, f, indent=2)

    dataset_json = {
        "channel_names": {"0": "grayscale"},
        "labels": {"background": 0, **{name: int(i) for i, name in nnunet_to_name.items() if i != 0}},
        "numTraining": int(len(train_cases)),
        "file_ending": ".png",
    }
    with open(dataset_root / "dataset.json", "w") as f:
        json.dump(dataset_json, f, indent=2)

    class_mapping = {
        "atlas_to_nnunet": {int(k): int(v) for k, v in atlas_to_nnunet.items()},
        "nnunet_to_name_placeholder": nnunet_to_name,
        "note": "Names are placeholders (structure_<atlas_id>) until a doctor names each ID.",
    }
    with open(dataset_root / "class_mapping.json", "w") as f:
        json.dump(class_mapping, f, indent=2)

    # --- package + push this group's dataset to HF ---
    package_root = ROOT / f"hf_package_group{group_index:02d}"
    if package_root.exists():
        shutil.rmtree(package_root)
    package_root.mkdir()
    shutil.copytree(dataset_root, package_root / dataset_root.name)

    num_images = len(list(images_dir.glob("*.png")))
    num_labels = len(list(labels_dir.glob("*.png")))
    assert num_images == num_labels == len(train_cases), \
        f"[DATA PROBLEM] group {group_index}: image/label/case-count mismatch: " \
        f"{num_images}/{num_labels}/{len(train_cases)}"

    api.upload_folder(
        folder_path=str(package_root / dataset_root.name),
        path_in_repo=f"training_dataset/{SPECIMEN_ID}/{dataset_root.name}",
        repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, token=HF_TOKEN,
    )
    print(f"Pushed group {group_index} -> https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/"
          f"training_dataset/{SPECIMEN_ID}/{dataset_root.name}")

    all_groups_manifest["groups"].append({
        "group_index": group_index,
        "dataset_id": dataset_id,
        "dataset_name": dataset_name,
        "dataset_folder": dataset_root.name,
        "n_classes": len(sorted_group_ids),
        "atlas_ids": sorted_group_ids,
        "total_positive_pixels": int(sum(size_by_id[a] for a in sorted_group_ids)),
        "n_train_cases": len(train_cases),
        "n_holdout_cases": len(holdout_cases),
    })

print(f"\nAll {len(all_groups_manifest['groups'])} group datasets built and pushed.")


Fresh HF dataset repo confirmed empty: senku111/atlas-cs17-4groups-dataset

GROUP 0  ->  Dataset701_AtlasCS17FourGroup00  (21 classes)
1056 non-empty slices -> train=845, holdout=211
Verified: 0 of 211 held-out cases present in imagesTr/labelsTr.


It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed group 0 -> https://huggingface.co/datasets/senku111/atlas-cs17-4groups-dataset/tree/main/training_dataset/CS17-6521/Dataset701_AtlasCS17FourGroup00

GROUP 1  ->  Dataset702_AtlasCS17FourGroup01  (22 classes)
Verified: 0 of 153 held-out cases present in imagesTr/labelsTr.


...

Pushed group 2 -> https://huggingface.co/datasets/senku111/atlas-cs17-4groups-dataset/tree/main/training_dataset/CS17-6521/Dataset703_AtlasCS17FourGroup02

GROUP 3  ->  Dataset704_AtlasCS17FourGroup03  (22 classes)
540 non-empty slices -> train=432, holdout=108
Verified: 0 of 108 held-out cases present in imagesTr/labelsTr.


It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed group 3 -> https://huggingface.co/datasets/senku111/atlas-cs17-4groups-dataset/tree/main/training_dataset/CS17-6521/Dataset704_AtlasCS17FourGroup03

All 4 group datasets built and pushed.


## 7. Push the top-level groups manifest

`all_groups_manifest.json` is the single source of truth for "which groups exist, and where".
The training notebook downloads this file first, then uses it to resolve a single `GROUP_INDEX`
variable into the right dataset ID/name/folder -- that's the mechanism that lets one training
notebook produce a separate model per group just by changing one variable.


In [7]:
# ============================================================
# 7. PUSH THE TOP-LEVEL GROUPS MANIFEST + REPORTS
# ============================================================
manifest_path = REPORT_DIR / "all_groups_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(all_groups_manifest, f, indent=2)

for report in REPORT_DIR.glob("*"):
    if report.is_file():
        api.upload_file(
            path_or_fileobj=str(report),
            path_in_repo=f"training_dataset/{SPECIMEN_ID}/reports/{report.name}",
            repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, token=HF_TOKEN,
        )

print(f"UPLOAD COMPLETE: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/training_dataset/{SPECIMEN_ID}")
print(f"Manifest: https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/training_dataset/"
      f"{SPECIMEN_ID}/reports/all_groups_manifest.json\n")
for g in all_groups_manifest["groups"]:
    print(f"  group {g['group_index']:2d}: Dataset{g['dataset_id']:03d}_{g['dataset_name']} -- "
          f"{g['n_classes']} classes, {g['n_train_cases']} train / {g['n_holdout_cases']} holdout slices")


UPLOAD COMPLETE: https://huggingface.co/datasets/senku111/atlas-cs17-4groups-dataset/tree/main/training_dataset/CS17-6521
Manifest: https://huggingface.co/datasets/senku111/atlas-cs17-4groups-dataset/blob/main/training_dataset/CS17-6521/reports/all_groups_manifest.json

  group  0: Dataset701_AtlasCS17FourGroup00 -- 21 classes, 845 train / 211 holdout slices
  group  1: Dataset702_AtlasCS17FourGroup01 -- 22 classes, 615 train / 153 holdout slices
  group  2: Dataset703_AtlasCS17FourGroup02 -- 22 classes, 697 train / 174 holdout slices
  group  3: Dataset704_AtlasCS17FourGroup03 -- 22 classes, 432 train / 108 holdout slices


# DONE

The NEW Hugging Face dataset repo (`senku111/atlas-cs17-4groups-dataset`) now contains, for EVERY
one of the `N_GROUPS` groups:

- `Dataset{ID}_{NAME}/imagesTr`, `labelsTr` -- that group's nnU-Net **training** set only
- `Dataset{ID}_{NAME}/heldout/images`, `heldout/labels` -- physically separate slices nnU-Net's
  own planning, preprocessing, and cross-validation never see
- `Dataset{ID}_{NAME}/dataset.json` (`numTraining` matches `imagesTr` exactly)
- `Dataset{ID}_{NAME}/class_mapping.json` -- Atlas ID <-> nnU-Net ID <-> placeholder name
- `Dataset{ID}_{NAME}/split_manifest.json` -- fixed seed + exact case IDs in each group's split

...plus, once per specimen:

- `reports/all_groups_manifest.json` -- every group, its dataset ID/name/folder, Atlas IDs and
  train/holdout counts. **The training notebook reads this file.**
- `reports/class_representation_stats.csv`, `reports/groups_summary.csv`

**Class-selection reminder:** nothing was filtered. Every structure ID present in the labels TIFF
is in exactly one group's training set. Names are placeholders (`structure_<atlas_id>`) until a
doctor names them from the trained models' predictions.

**Grouping reminder:** exactly `N_GROUPS` groups, near-equal class counts, and structures of
similar size inside each group so no single class dominates a model.

**Normalization reminder:** all exported slices, in every group, share one global intensity range.

The GPU training notebook downloads this repository and trains one model per group, selected by a
single `GROUP_INDEX` variable (0 ... N_GROUPS-1).
